# ML Model Evaluation & Results
**DATA 228 — Spring 2026 | Team 3**

Comprehensive evaluation of all 4 ML models:
1. CTR Prediction (LR vs GBT)
2. ALS Collaborative Filtering vs Popularity Baseline
3. Product2Vec Session-Based Recommendations
4. K-Means Customer Segmentation

---

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

ML_DIR = os.path.join('..', 'ml_artifacts')

def load_results(filename):
    path = os.path.join(ML_DIR, filename)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

print('Loading ML results...')

## 1. CTR Prediction — Logistic Regression vs GBT

In [ ]:
ctr_results = load_results('ctr_comparison.json')

if ctr_results:
    metrics = ['auc_roc', 'auc_pr', 'accuracy', 'f1_score']
    lr = ctr_results['logistic_regression']
    gbt = ctr_results['gbt']
    
    fig = go.Figure()
    fig.add_trace(go.Bar(name='Logistic Regression',
                         x=metrics, y=[lr[m] for m in metrics],
                         marker_color='#636EFA', text=[f"{lr[m]:.4f}" for m in metrics],
                         textposition='outside'))
    fig.add_trace(go.Bar(name='Gradient Boosted Trees',
                         x=metrics, y=[gbt[m] for m in metrics],
                         marker_color='#EF553B', text=[f"{gbt[m]:.4f}" for m in metrics],
                         textposition='outside'))
    fig.update_layout(barmode='group', title='CTR Model Comparison',
                      yaxis_title='Score', height=450)
    fig.show()
    
    # Training time comparison
    time_df = pd.DataFrame({
        'Model': ['LR', 'GBT'],
        'Training Time (s)': [lr['training_time_sec'], gbt['training_time_sec']],
        'AUC-ROC': [lr['auc_roc'], gbt['auc_roc']]
    })
    fig2 = px.scatter(time_df, x='Training Time (s)', y='AUC-ROC', text='Model',
                      size=[30, 30], title='Accuracy vs Compute Tradeoff',
                      color='Model')
    fig2.update_traces(textposition='top center')
    fig2.show()
else:
    print('Run: spark-submit ml/ctr/ctr_model.py')

## 2. ALS Recommender vs Popularity Baseline

In [ ]:
als_results = load_results('als_results.json')

if als_results:
    als = als_results['als']
    pop = als_results['popularity_baseline']
    
    metrics = [k for k in als.keys() if k.startswith('precision') or k.startswith('recall')]
    
    fig = go.Figure()
    fig.add_trace(go.Bar(name='ALS', x=metrics,
                         y=[als[m] for m in metrics], marker_color='#AB63FA'))
    fig.add_trace(go.Bar(name='Popularity', x=metrics,
                         y=[pop.get(m, 0) for m in metrics], marker_color='#888'))
    fig.update_layout(barmode='group', title='ALS vs Popularity Baseline',
                      yaxis_title='Score', height=400)
    fig.show()
    
    # Compute lift
    for m in metrics:
        als_val = als[m]
        pop_val = pop.get(m, 0)
        lift = ((als_val / pop_val) - 1) * 100 if pop_val > 0 else float('inf')
        print(f'{m}: ALS={als_val:.4f} vs Pop={pop_val:.4f} → Lift: {lift:+.1f}%')
    
    ndcg = als.get('ndcg@10', 'N/A')
    print(f'\nNDCG@10: {ndcg}')
    print(f'Config: rank={als_results["config"]["rank"]}, '
          f'reg={als_results["config"]["reg_param"]}')
else:
    print('Run: spark-submit ml/recommender/als_recommender.py')

## 3. Product2Vec Evaluation

In [ ]:
p2v_results = load_results('product2vec_results.json')

if p2v_results:
    metrics = p2v_results['metrics']
    config = p2v_results['config']
    
    fig = go.Figure()
    fig.add_trace(go.Indicator(
        mode='gauge+number',
        value=metrics['hit_rate@10'] * 100,
        title={'text': 'Hit Rate @10 (%)'},
        gauge={'axis': {'range': [0, 50]},
               'bar': {'color': '#00CC96'},
               'steps': [
                   {'range': [0, 10], 'color': '#fee2e2'},
                   {'range': [10, 25], 'color': '#fef9c3'},
                   {'range': [25, 50], 'color': '#d1fae5'},
               ]},
        domain={'x': [0, 0.45], 'y': [0, 1]}
    ))
    fig.add_trace(go.Indicator(
        mode='gauge+number',
        value=metrics['coverage'] * 100,
        title={'text': 'Coverage (%)'},
        gauge={'axis': {'range': [0, 100]},
               'bar': {'color': '#636EFA'}},
        domain={'x': [0.55, 1], 'y': [0, 1]}
    ))
    fig.update_layout(height=300, title='Product2Vec Performance')
    fig.show()
    
    print(f"Vector size: {config['vector_size']}")
    print(f"Vocab size: {metrics['vocab_size']:,} products")
    print(f"Training time: {p2v_results['training_time_sec']}s")
else:
    print('Run: spark-submit ml/session/product2vec.py')

## 4. Customer Segmentation

In [ ]:
seg_results = load_results('segmentation_results.json')

if seg_results:
    # K selection plot
    k_data = seg_results['k_search_results']
    k_df = pd.DataFrame([
        {'K': int(k), 'Silhouette': v['silhouette'], 'Inertia': v['inertia']}
        for k, v in k_data.items()
    ]).sort_values('K')
    
    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Scatter(x=k_df['K'], y=k_df['Silhouette'],
                             name='Silhouette', mode='lines+markers',
                             marker=dict(size=10)), secondary_y=False)
    fig.add_trace(go.Scatter(x=k_df['K'], y=k_df['Inertia'],
                             name='Inertia', mode='lines+markers',
                             marker=dict(size=10), line=dict(color='#EF553B')),
                  secondary_y=True)
    fig.update_layout(title=f'Optimal K Selection (Best K = {seg_results["optimal_k"]})',
                      height=400)
    fig.update_yaxes(title_text='Silhouette Score', secondary_y=False)
    fig.update_yaxes(title_text='Inertia', secondary_y=True)
    fig.show()
    
    # Segment labels
    labels = seg_results.get('segment_labels', {})
    print('\nSegment Labels:')
    for k, v in labels.items():
        print(f'  Cluster {k}: {v}')
else:
    print('Run: spark-submit ml/segmentation/customer_segments.py')

## 5. Cross-Model Summary

In [ ]:
summary = []

if ctr_results:
    summary.append({'Model': 'CTR - LR', 'Primary Metric': 'AUC-ROC',
                    'Score': ctr_results['logistic_regression']['auc_roc'],
                    'Dataset': 'Criteo', 'Time (s)': ctr_results['logistic_regression']['training_time_sec']})
    summary.append({'Model': 'CTR - GBT', 'Primary Metric': 'AUC-ROC',
                    'Score': ctr_results['gbt']['auc_roc'],
                    'Dataset': 'Criteo', 'Time (s)': ctr_results['gbt']['training_time_sec']})

if als_results:
    summary.append({'Model': 'ALS Recommender', 'Primary Metric': 'Precision@10',
                    'Score': als_results['als']['precision@10'],
                    'Dataset': 'REES46', 'Time (s)': als_results['als']['training_time_sec']})

if p2v_results:
    summary.append({'Model': 'Product2Vec', 'Primary Metric': 'Hit Rate@10',
                    'Score': p2v_results['metrics']['hit_rate@10'],
                    'Dataset': 'REES46', 'Time (s)': p2v_results['training_time_sec']})

if summary:
    summary_df = pd.DataFrame(summary)
    display(summary_df)
    
    fig = px.bar(summary_df, x='Model', y='Score', color='Dataset',
                 text='Score', title='All Models — Primary Metric Comparison')
    fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
    fig.update_layout(height=400)
    fig.show()
else:
    print('No model results found. Run: make ml')